# 🧪 Lab 02: Processamento de Linguagem Natural (NLP) com LSTM no PyTorch
**Módulo:** `02-deep-learning / computer-vision-nlp`

## Objetivos
1. Pré-processar dados de texto: tokenização, criação de **vocabulário** e **padding**.
2. Construir uma camada de **Word Embeddings** (`nn.Embedding`).
3. Criar uma rede recorrente **LSTM (Long Short-Term Memory)** para Análise de Sentimento (Classificação Binária).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Verificar dispositivo (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executando no dispositivo: {device}")

Executando no dispositivo: cpu


## 1. Criação e Pré-processamento de um Dataset de Texto Sintético

In [2]:
# Frases de exemplo (1 = Positivo, 0 = Negativo)
corpus = [
    ("excelente produto muito bom recomendo", 1),
    ("pessimo atendimento nao gostei", 0),
    ("ótima qualidade entrega rapida", 1),
    ("produto com defeito muito ruim", 0),
    ("servico fantastico super satisfeito", 1),
    ("horrivel nao vale o preço", 0)
]

# Construção do Vocabulário
words = set(" ".join([text for text, _ in corpus]).split())
vocab = {word: idx + 2 for idx, word in enumerate(words)}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

vocab_size = len(vocab)
max_len = 6

def text_to_pipeline(text):
    tokens = text.split()
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    # Padding
    if len(ids) < max_len:
        ids += [vocab["<PAD>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return ids

X_data = torch.tensor([text_to_pipeline(text) for text, _ in corpus], dtype=torch.long)
y_data = torch.tensor([label for _, label in corpus], dtype=torch.float32).unsqueeze(1)

dataset = TensorDataset(X_data, y_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print("Formato dos dados de entrada (X):", X_data.shape)
print("Vocabulário criado com tamanho:", vocab_size)

Formato dos dados de entrada (X): torch.Size([6, 6])
Vocabulário criado com tamanho: 26


## 2. Arquitetura da Rede Recorrente (LSTM Classifier)

In [3]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)                     # [batch_size, seq_len, embed_dim]
        _, (hidden, _) = self.lstm(embedded)             # hidden: [1, batch_size, hidden_dim]
        out = self.fc(hidden.squeeze(0))                 # [batch_size, output_dim]
        return self.sigmoid(out)

model = SentimentLSTM(vocab_size=vocab_size, embedding_dim=16, hidden_dim=32, output_dim=1).to(device)
print(model)

SentimentLSTM(
  (embedding): Embedding(26, 16, padding_idx=0)
  (lstm): LSTM(16, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 3. Treino e Teste em Novas Frases

In [4]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 50
for epoch in range(epochs):
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()

print("✅ Treinamento da LSTM concluído!")

# Teste com uma frase inédita
def prever_sentimento(frase):
    model.eval()
    with torch.no_grad():
        ids = torch.tensor([text_to_pipeline(frase)], dtype=torch.long).to(device)
        prob = model(ids).item()
        sentimento = "Positivo 😊" if prob > 0.5 else "Negativo 😞"
        print(f"Frase: '{frase}' -> Probabilidade: {prob:.4f} | Sentimento: {sentimento}")

prever_sentimento("excelente qualidade super bom")
prever_sentimento("pessimo produto horrivel")

✅ Treinamento da LSTM concluído!
Frase: 'excelente qualidade super bom' -> Probabilidade: 0.0008 | Sentimento: Negativo 😞
Frase: 'pessimo produto horrivel' -> Probabilidade: 0.0003 | Sentimento: Negativo 😞
